# Revised Figure 3: hapax rate vs. entity-level F1 (no GPU)

Run after notebook 07, which made `results/bootstrap_summary.json`. Use a **CPU** runtime and click **Run all**.

Same style as the original figure, with the revised numbers:
* hapax rates from **Table 2** (72.0 / 80.0 / 82.4 / 86.9%)
* F1 of the best system, **XLM-R-large-ID-NER + CRF**, as the mean of 3 seeds, with SD error bars

`TREND = False` by default. The revised text says hapax rate alone does not order SYM, TMP and RF, so a trend line would contradict it. Set `TREND = True` only if you and your supervisor decide to keep it.

Output in `MyDrive/strokeid_revision/results/`: `figure3.pdf` (upload this to the journal) and `figure3.png` (600 dpi).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS = '/content/drive/MyDrive/strokeid_revision/results'
assert os.path.exists(f'{RESULTS}/bootstrap_summary.json'), 'run notebook 07 first'
os.makedirs('/content/code', exist_ok=True)
TREND = False      # True = draw the dashed DGN/SYM/TMP trend line of the original figure
print('OK')

In [ ]:
%%writefile /content/code/make_fig3_styled.py
"""Figure 3 in the style of the original manuscript figure, with revised numbers.

Reads results/bootstrap_summary.json (notebook 07) and plots, per entity type,
the best system's F1 (XLM-R-large-ID-NER + CRF, mean over 3 seeds; thin error bar
= SD) against the hapax rate from Table 2.

Options
  --trend   draw the dashed DGN/SYM/TMP trend line of the original figure
            (off by default: the revised text says hapax rate alone does not
            order SYM, TMP and RF)
  --model   which system to plot (default m3crf)

Usage:  python make_fig3_styled.py --summary results/bootstrap_summary.json --out results/figure3
"""
import argparse
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HAPAX = {"DGN": 72.0, "SYM": 80.0, "TMP": 82.4, "RF": 86.9}            # Table 2 (case-sensitive)
COLORS = {"DGN": "#2E75B6", "SYM": "#C00000", "TMP": "#375623", "RF": "#7030A0"}  # original figure
NAMES = {"m3crf": "XLM-R-large-ID-NER + CRF", "m3": "XLM-R-large-ID-NER", "m5crf": "XLM-R-large + CRF"}
# label box offsets in points (dx, dy), chosen so boxes do not overlap points or each other
OFFSETS = {"DGN": (12, 14), "SYM": (-12, -34), "TMP": (-150, 22), "RF": (-40, -48)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--summary", required=True)
    ap.add_argument("--out", default="figure3")
    ap.add_argument("--model", default="m3crf")
    ap.add_argument("--trend", action="store_true")
    a = ap.parse_args()
    s = json.load(open(a.summary))["models"][a.model]

    fig, ax = plt.subplots(figsize=(7.8, 5.9))
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, color="#e6e6e6", linewidth=0.8)
    ax.set_axisbelow(True)

    if a.trend:
        xs = np.array([HAPAX[e] for e in ("DGN", "SYM", "TMP")])
        ys = np.array([s["mean"][e] for e in ("DGN", "SYM", "TMP")])
        k, b = np.polyfit(xs, ys, 1)
        gx = np.array([68, 84])
        ax.plot(gx, k * gx + b, "--", color="#bdbdbd", linewidth=1.8, label="Trend (DGN/SYM/TMP)", zorder=1)
        ax.legend(loc="upper right", fontsize=10)

    for e, x in HAPAX.items():
        y, sd = s["mean"][e], s["sd"][e]
        c = COLORS[e]
        ax.errorbar(x, y, yerr=sd, fmt="none", ecolor=c, elinewidth=1.2, capsize=4, zorder=2)
        ax.scatter(x, y, s=230, color=c, edgecolors="white", linewidths=1.5, zorder=3)
        dx, dy = OFFSETS[e]
        ax.annotate(f"{e}\nhapax={x:.1f}%  F1={y:.3f}", (x, y), xytext=(dx, dy), textcoords="offset points",
                    fontsize=11, fontweight="bold", color=c, va="center", ha="left",
                    bbox=dict(boxstyle="round,pad=0.35", fc="white", ec=c, lw=1.2, alpha=0.95), zorder=4)

    ax.set_xlim(66, 92)
    ax.set_ylim(0.55, 0.95)
    ax.set_xlabel("Hapax Rate (%)", fontsize=12)
    ax.set_ylabel(f"F1 Score ({NAMES.get(a.model, a.model)})", fontsize=12)
    ax.set_title("Hapax Rate vs Entity-Level F1", fontsize=13, fontweight="bold")
    ax.tick_params(labelsize=11)
    ax.text(0.02, 0.03, "Points: mean of 3 seeds; error bars: SD. Hapax rates from Table 2.",
            transform=ax.transAxes, fontsize=9, style="italic", color="#808080")
    fig.tight_layout()
    fig.savefig(a.out + ".pdf")
    fig.savefig(a.out + ".png", dpi=600)
    print("written", a.out + ".pdf", a.out + ".png")


if __name__ == "__main__":
    main()


In [ ]:
flag = '--trend' if TREND else ''
!cd /content/code && python make_fig3_styled.py --summary "$RESULTS/bootstrap_summary.json" --out "$RESULTS/figure3" $flag
from IPython.display import Image
Image(f'{RESULTS}/figure3.png', width=650)